In [3]:
from tensorflow.python.keras.layers import Dense,Conv2D,Flatten,MaxPooling2D,ZeroPadding2D,Dropout,Softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.python.keras import Sequential
from sklearn.datasets import fetch_lfw_people
from sklearn.datasets import fetch_lfw_pairs
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import scikitplot as skplt
import numpy as np
import random

# Commented out IPython magic to ensure Python compatibility.
from __future__ import absolute_import, division, print_function, unicode_literals
import tensorflow_datasets as tfds
import tensorflow as tf
tf.test.gpu_device_name()

data=fetch_lfw_people(resize=0.4,min_faces_per_person=40,funneled=False)

l=list(data.keys())
X=data[l[0]]
Y=data[l[2]]
target_name=data[l[3]]

_,W,H=data[l[1]].shape
features=X.shape[1]
m=X.shape[0]
classes=data[l[3]].shape[0]

print("Samples:",m)
print("Features:",features)
print("Classes:",classes)
print("Dimension:",(W,H))

### Shuffling the data ###

temp=list(zip(X,Y)) 
random.shuffle(temp) 
X,Y=zip(*temp)

X=np.array(X)
Y=np.array(Y)

Samples: 905
Features: 1850
Classes: 6
Dimension: (50, 37)
[[0.         0.         0.         ... 0.1006536  0.07712419 0.5111111 ]
 [0.         0.         0.         ... 0.3503268  0.36601308 0.37254903]
 [0.23137255 0.23006536 0.26013073 ... 0.28235295 0.27450982 0.27450982]
 ...
 [0.         0.         0.         ... 0.04705882 0.01568628 0.02745098]
 [0.         0.         0.         ... 0.03267974 0.03267974 0.03660131]
 [0.04705882 0.06535948 0.08366013 ... 0.09019608 0.06666667 0.04575163]]
hola
[1 5 1 2 1 1 1 0 4 2 1 1 0 1 1 0 1 1 1 1 1 2 2 1 3 2 1 0 2 1 1 1 3 1 1 0 1
 1 5 4 0 1 2 1 1 1 0 1 1 2 2 2 1 1 1 1 1 1 1 2 1 1 3 1 2 4 1 1 4 1 1 0 1 1
 5 1 1 1 1 3 1 4 1 1 1 1 1 1 4 2 1 2 3 0 2 1 1 4 1 2 1 1 1 1 1 1 1 1 1 1 1
 5 1 1 3 0 1 1 0 4 5 1 1 3 1 5 1 0 1 2 1 1 0 0 1 1 1 1 1 1 5 2 1 4 1 5 2 1
 2 3 1 1 1 0 1 1 0 1 2 1 1 1 4 1 1 1 0 1 3 5 1 1 0 1 2 3 0 0 2 1 0 0 1 1 3
 2 1 1 1 0 3 1 2 1 1 2 1 1 1 1 1 0 1 1 1 1 2 1 1 1 2 1 0 1 4 1 5 1 2 5 2 1
 1 0 0 1 3 1 1 1 0 5 0 1 3 0 1 1 1 1 4 2 2

2022-10-23 21:46:32.910450: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2022-10-23 21:46:32.910475: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [4]:
####### Train and test splitting of data #######

training_data_X,testing_data_X,training_data_Y,testing_data_Y=train_test_split(X,Y,test_size=0.20,random_state=42)

####### CNN #######
model=Sequential()

model.add(Conv2D(64,kernel_size = 3,activation = 'relu',input_shape = (W,H,1)))
model.add(MaxPooling2D((2,2),strides = (2,2)))
model.add(Conv2D(128,kernel_size = 3,activation = 'relu'))
model.add(MaxPooling2D((2,2),strides = (2,2)))
model.add(Conv2D(256,kernel_size = 5,activation = 'relu'))
model.add(MaxPooling2D((2,2),strides = (2,2)))
model.add(Dropout(0.25))
#model.add(Conv2D(512,kernel_size = 3,activation = 'relu'))
#model.add(MaxPooling2D((2,2),strides = (2,2)))

model.add(Flatten())
model.add(Dense(classes,activation = 'softmax'))

from keras.utils import to_categorical

training_data_Y = to_categorical(training_data_Y)
testing_data_Y = to_categorical(testing_data_Y)

training_data_X = training_data_X.reshape(training_data_X.shape[0],W,H,1)
testing_data_X = testing_data_X.reshape(testing_data_X.shape[0],W,H,1)

import time
# Measuring the time taken by the model to train
start_time = time.time()

model.compile(optimizer='Adamax',loss='categorical_crossentropy',metrics=['accuracy'])
history = model.fit(training_data_X,training_data_Y,validation_data = (testing_data_X, testing_data_Y),batch_size = 100,epochs = 50)

end_time = time.time()
print("###### Total Time Taken: ", round((end_time - start_time)/60), 'Minutes ######')

score = model.evaluate(testing_data_X,testing_data_Y,verbose=0)
print(score[1]*100)

Epoch 1/50


InvalidArgumentError: Cannot assign a device for operation sequential_1/conv2d_3/Conv2D/ReadVariableOp: Could not satisfy explicit device specification '' because the node {{colocation_node sequential_1/conv2d_3/Conv2D/ReadVariableOp}} was colocated with a group of nodes that required incompatible device '/job:localhost/replica:0/task:0/device:GPU:0'. All available devices [/job:localhost/replica:0/task:0/device:CPU:0, /job:localhost/replica:0/task:0/device:GPU:0]. 
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=2 requested_device_name_='/job:localhost/replica:0/task:0/device:GPU:0' assigned_device_name_='/job:localhost/replica:0/task:0/device:GPU:0' resource_device_name_='/job:localhost/replica:0/task:0/device:GPU:0' supported_device_types_=[CPU] possible_devices_=[]
ResourceApplyAdaMax: CPU 
ReadVariableOp: GPU CPU 
_Arg: GPU CPU 

Colocation members, user-requested devices, and framework assigned device

In [ ]:
accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(accuracy) + 1)
plt.plot(epochs, accuracy, "bo", label  ="Training accuracy") 
plt.plot(epochs, val_accuracy, "b", label = "Validation accuracy") 
plt.title("Training and validation accuracy")
plt.legend()
plt.figure()
plt.plot(epochs, loss, "bo", label = "Training loss") 
plt.plot(epochs, val_loss, "b", label = "Validation loss") 
plt.title("Training and validation loss")
plt.legend()
plt.show()